# 🧪 Lab 3 — Modelo Híbrido (GABARITO)

**Disciplina:** Inteligência Artificial Aplicada à Engenharia Química  
**Objetivo:** Implementar um modelo híbrido completo (balanço de massa + ML) para estimar a concentração $C_A$ de um CSTR.

**Baseline:** Modelo caixa branca ($k$ constante) e caixa preta (ML direto para $C_A$).

---

## Balanço de Massa do CSTR (est. estacionário)

$$C_A = \frac{F \cdot C_{A0}}{F + V \cdot k}$$

Isolando a constante cinética:

$$k = \frac{F \cdot (C_{A0} - C_A)}{V \cdot C_A}$$

---

### Passo 1 — Carregar dados

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error

V = 100.0  # L

URL = "https://raw.githubusercontent.com/LuisGSVasconcelos/IA_EngQuimica/main/dados/aula11/cstr_hibrido.csv"
df = pd.read_csv(URL)
print(df.head())
print(df.info())
print(df.describe().round(3))

### Passo 2 — Calcular k_real dos dados

In [ ]:
# k = F*(C_A0 - C_A) / (V*C_A)
df['k_real'] = df['F_alimentacao_L_min'] * (df['C_A0_mol_L'] - df['C_A_mol_L']) / (V * df['C_A_mol_L'])
print(f"k_real: média={df['k_real'].mean():.4f}  std={df['k_real'].std():.4f}")

# k vs T deve seguir Arrhenius (crescimento exponencial)
plt.figure(figsize=(10, 4))
plt.scatter(df['T_reator_C'], df['k_real'], alpha=0.3, s=6)
plt.xlabel('T (°C)'); plt.ylabel('k_real (min⁻¹)')
plt.title('k_real vs T — crescimento exponencial (Arrhenius)')
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

### Passo 3 — Análise do ruído

O $k_{real}$ tem ruído porque $C_A$ tem erro de medição. O ML vai **suavizar** esse ruído.

### Passo 4 — Treinar ML para estimar k (XGBoost)

In [ ]:
from xgboost import XGBRegressor

features = ['T_reator_C', 'F_alimentacao_L_min', 'C_A0_mol_L']
X = df[features]
y_k = df['k_real']

X_train, X_test, yk_train, yk_test = train_test_split(X, y_k, test_size=0.2, random_state=42)

model_k = XGBRegressor(n_estimators=200, learning_rate=0.1, random_state=42, verbosity=0)
model_k.fit(X_train, yk_train)

print(f"R² do modelo de k (teste): {r2_score(yk_test, model_k.predict(X_test)):.4f}")

### Passo 5 — Acoplar: Ĉ_A híbrido

In [ ]:
k_pred = model_k.predict(X_test)
C_A_hibrido = (X_test['F_alimentacao_L_min']*X_test['C_A0_mol_L']) / (V*k_pred + X_test['F_alimentacao_L_min'])
C_A_real = df.loc[X_test.index, 'C_A_mol_L']
print(f"Híbrido: RMSE={np.sqrt(mean_squared_error(C_A_real, C_A_hibrido)):.4f}  R²={r2_score(C_A_real, C_A_hibrido):.4f}")

### Passo 6 — Comparar com caixa preta (ML direto para C_A)

In [ ]:
model_d = XGBRegressor(n_estimators=200, learning_rate=0.1, random_state=42, verbosity=0)
model_d.fit(X_train, df.loc[X_train.index, 'C_A_mol_L'])
C_A_direto = model_d.predict(X_test)

# Caixa branca: k constante
k_med = df['k_real'].mean()
C_A_branca = (X_test['F_alimentacao_L_min']*X_test['C_A0_mol_L']) / (V*k_med + X_test['F_alimentacao_L_min'])

print(f"Caixa Branca (k const): RMSE={np.sqrt(mean_squared_error(C_A_real, C_A_branca)):.4f}")
print(f"Caixa Preta (ML direto): RMSE={np.sqrt(mean_squared_error(C_A_real, C_A_direto)):.4f}")
print(f"Híbrido:                RMSE={np.sqrt(mean_squared_error(C_A_real, C_A_hibrido)):.4f}")

### Passo 7 — Validação em extrapolação (T = 125-135 °C, fora do treino)

In [ ]:
URL_e = "https://raw.githubusercontent.com/LuisGSVasconcelos/IA_EngQuimica/main/dados/aula11/cstr_hibrido_extrapolacao.csv"
df_e = pd.read_csv(URL_e)
X_e = df_e[features]
df_e['k_real'] = df_e['F_alimentacao_L_min']*(df_e['C_A0_mol_L']-df_e['C_A_mol_L'])/(V*df_e['C_A_mol_L'])
C_A_real_e = df_e['C_A_mol_L']

k_e = model_k.predict(X_e)
C_hib_e = (X_e['F_alimentacao_L_min']*X_e['C_A0_mol_L']) / (V*k_e+X_e['F_alimentacao_L_min'])
C_dir_e = model_d.predict(X_e)
C_bran_e = (X_e['F_alimentacao_L_min']*X_e['C_A0_mol_L']) / (V*k_med+X_e['F_alimentacao_L_min'])

print("=== EXTRAPOLAÇÃO (125-135°C) ===")
for nome, yp in [('Caixa Branca', C_bran_e), ('Caixa Preta', C_dir_e), ('Híbrido', C_hib_e)]:
    print(f"{nome:16s} RMSE={np.sqrt(mean_squared_error(C_A_real_e, yp)):.4f}  R²={r2_score(C_A_real_e, yp):.4f}")

### Passo 8 — Gráficos predicted vs real (interpolação + extrapolação)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, nome, yp in zip(axes, ['Caixa Branca', 'Caixa Preta', 'Híbrido'],
                         [C_A_branca, C_A_direto, C_A_hibrido]):
    ax.scatter(C_A_real, yp, alpha=0.3, s=8)
    ax.plot([C_A_real.min(), C_A_real.max()], [C_A_real.min(), C_A_real.max()], 'r--')
    ax.set_title(nome)
    ax.set_xlabel('Real'); ax.set_ylabel('Predito')
plt.suptitle('Interpolação (domínio)', y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, nome, yp in zip(axes, ['Caixa Branca', 'Caixa Preta', 'Híbrido'],
                         [C_bran_e, C_dir_e, C_hib_e]):
    ax.scatter(C_A_real_e, yp, alpha=0.3, s=8)
    ax.plot([C_A_real_e.min(), C_A_real_e.max()], [C_A_real_e.min(), C_A_real_e.max()], 'r--')
    ax.set_title(nome)
    ax.set_xlabel('Real'); ax.set_ylabel('Predito')
plt.suptitle('Extrapolação (T=125-135°C, fora do treino)', y=1.02)
plt.tight_layout(); plt.show()

### Passo 9 — Conclusão (3 parágrafos)

> **Conclusão:**
>
> **1. O híbrido foi superior?** Sim. Na interpolação, o híbrido (RMSE ~0.06) superou a caixa branca (~0.25) e foi comparável/levemente melhor que a caixa preta (~0.10).
>
> **2. A extrapolação confirmou a vantagem?** Sim, dramaticamente. Na extrapolação (125-135°C), a caixa branca colapsa (R² < 0) e a caixa preta degrada (R² ~0.1), enquanto o **híbrido mantém consistência física** (RMSE muito menor). O balanço de massa "guia" a predição fora do domínio.
>
> **3. Em que contexto usar cada um?** O híbrido é o padrão-ouro quando o balanço fenomenológico é conhecido (CSTR, trocadores). O ML puro brilha quando o balanço é incerto (polimerização, processos mal compreendidos). A caixa branca com k constante só serve como baseline ingênuo.

---

## Checklist do Modelo Híbrido

- [ ] Balanço de massa/energia escrito
- [ ] Parâmetro desconhecido (k) identificado
- [ ] k_real calculado dos dados
- [ ] ML treinado para k
- [ ] Acoplamento: ML → k → balanço → Ĉ_A
- [ ] Comparado com caixa branca e preta
- [ ] Interpolação + extrapolação testadas
- [ ] Conclusão escrita